In [1]:
# %% [markdown]
# # Paper #13 — Reasoning-State Expression Pipeline
# **Blueprint version:** v2.0 Final
# **Pipeline version:** v1.0 (frozen, engineering freeze — see PIPELINE_V1.0_FROZEN.md)
#
# This notebook is the single Kaggle execution artifact for:
# 1. Real tiktoken (o200k_base) tokenization of AgentErrorBench trajectories
# 2. Production of `predictor_table_tiktoken.csv` (the analysis-grade table)
# 3. Production of `predictor_table_diagnostic.csv` (whitespace-tokenizer, for comparison only)
# 4. A tokenizer comparison report quantifying that the swap changes scale, not extraction logic
#
# **Do not modify extraction/aggregation logic in this notebook.** Per the
# frozen Stage 4 governance rules, any change beyond the tokenizer itself
# constitutes a blueprint reopening and must be documented before proceeding.
#
# Confirmatory predictors (6): Memory Expression x3, Reflection Expression x3
# Exploratory-only predictors (+3, total 9): Planning Expression x3
# Removed (Step 3 reopening, see blueprint): Reasoning Expression (<think>-based),
# Reasoning-Action Balance

# %%
# Setup: install tiktoken (requires unrestricted network access -- this step
# is the entire reason this notebook runs on Kaggle rather than the dev sandbox)
!pip install -q tiktoken scipy scikit-bio

# %%
import json
import re
import csv
import statistics
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field

import tiktoken

# %% [markdown]
# ## 1. Tokenizer (frozen choice: tiktoken, o200k_base)

# %%
_encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    """FROZEN tokenizer per Step 3: tiktoken, o200k_base encoding."""
    if text is None:
        return 0
    return len(_encoding.encode(text))

def count_tokens_DIAGNOSTIC_ONLY(text):
    """NON-FROZEN. Whitespace-split approximation, comparison purposes only.
    Never used for the final predictor_table_tiktoken.csv."""
    if text is None:
        return 0
    return len(text.split())

print("tiktoken o200k_base loaded OK. Vocab size:", _encoding.n_vocab)

# %% [markdown]
# ## 2. Parser (unchanged from sandbox development; SHA-256 verified below)

# %%
TAG_NAMES = ["think", "memory", "reflection", "plan", "action", "answer"]
TAG_PATTERN = {tag: re.compile(rf"<{tag}>(.*?)</{tag}>", re.DOTALL) for tag in TAG_NAMES}


@dataclass
class StepExtraction:
    message_index: int
    think: str = None
    memory: str = None
    reflection: str = None
    plan: str = None
    action: str = None
    answer: str = None


@dataclass
class TrajectoryExtraction:
    trajectory_id: str
    environment: str
    file_path: str
    metadata: dict
    steps: list = field(default_factory=list)
    parse_errors: list = field(default_factory=list)


def extract_tags_from_content(content):
    result = {}
    for tag in TAG_NAMES:
        match = TAG_PATTERN[tag].search(content)
        result[tag] = match.group(1).strip() if match else None
    return result


def parse_trajectory_file(path, environment):
    trajectory_id = path.stem
    errors = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        errors.append(f"JSON_LOAD_FAILURE: {e}")
        return TrajectoryExtraction(trajectory_id, environment, str(path), {}, [], errors)

    metadata = data.get("metadata", {})
    if "metadata" not in data:
        errors.append("MISSING_METADATA_FIELD")
    messages = data.get("messages", [])
    if "messages" not in data:
        errors.append("MISSING_MESSAGES_FIELD")

    steps = []
    for i, msg in enumerate(messages):
        if msg.get("role") != "assistant":
            continue
        content = msg.get("content", "")
        if not isinstance(content, str):
            errors.append(f"NON_STRING_CONTENT_AT_INDEX_{i}")
            continue
        tags = extract_tags_from_content(content)
        steps.append(StepExtraction(
            message_index=i, think=tags["think"], memory=tags["memory"],
            reflection=tags["reflection"], plan=tags["plan"],
            action=tags["action"], answer=tags["answer"],
        ))

    return TrajectoryExtraction(trajectory_id, environment, str(path), metadata, steps, errors)


def parse_all_trajectories(data_dir):
    results = []
    for env_name in ["ALFWorld", "GAIA", "WebShop"]:
        env_dir = data_dir / env_name
        for jf in sorted(env_dir.glob("*.json")):
            results.append(parse_trajectory_file(jf, env_name))
    return results

# %% [markdown]
# ## 3. Aggregator (Blueprint v2: 9 predictors, Reasoning Expression and
# Reasoning-Action Balance removed per the Step 3 reopening)

# %%
@dataclass
class TrajectoryPredictors:
    trajectory_id: str
    environment: str
    model: str
    n_steps_total: int
    n_steps_with_memory: int
    n_steps_with_reflection: int
    n_steps_with_plan: int
    mean_memory_length: float = None
    memory_proportion: float = None
    memory_variability: float = None
    mean_reflection_length: float = None
    reflection_proportion: float = None
    reflection_variability: float = None
    mean_plan_length: float = None
    plan_proportion: float = None
    plan_variability: float = None
    extraction_warnings: list = field(default_factory=list)


def _lengths(steps, attr_name, tokenizer_fn):
    out = []
    for s in steps:
        val = getattr(s, attr_name)
        if val is not None:
            out.append(tokenizer_fn(val))
    return out


def compute_predictors(trajectory, tokenizer_fn):
    """tokenizer_fn is injected so the SAME aggregation logic can be run
    with either count_tokens (frozen, tiktoken) or count_tokens_DIAGNOSTIC_ONLY,
    producing the two comparison tables from identical code."""
    steps = trajectory.steps
    warnings_list = []

    memory_lens = _lengths(steps, "memory", tokenizer_fn)
    reflection_lens = _lengths(steps, "reflection", tokenizer_fn)
    plan_lens = _lengths(steps, "plan", tokenizer_fn)
    total_reasoning_tokens = sum(memory_lens) + sum(reflection_lens) + sum(plan_lens)

    def mean_or_none(lst):
        return statistics.mean(lst) if lst else None

    def var_or_none(lst):
        return statistics.variance(lst) if len(lst) >= 2 else None

    def proportion_or_none(component_sum, denom):
        return (component_sum / denom) if denom else None

    mean_memory = mean_or_none(memory_lens)
    memory_var = var_or_none(memory_lens)
    memory_prop = proportion_or_none(sum(memory_lens), total_reasoning_tokens)
    mean_reflection = mean_or_none(reflection_lens)
    reflection_var = var_or_none(reflection_lens)
    reflection_prop = proportion_or_none(sum(reflection_lens), total_reasoning_tokens)
    mean_plan = mean_or_none(plan_lens)
    plan_var = var_or_none(plan_lens)
    plan_prop = proportion_or_none(sum(plan_lens), total_reasoning_tokens)

    if not memory_lens:
        warnings_list.append("NO_MEMORY_TAGS_FOUND")
    if not reflection_lens:
        warnings_list.append("NO_REFLECTION_TAGS_FOUND")
    if not plan_lens:
        warnings_list.append("NO_PLAN_TAGS_FOUND")

    return TrajectoryPredictors(
        trajectory_id=trajectory.trajectory_id, environment=trajectory.environment,
        model=trajectory.metadata.get("model", "UNKNOWN"),
        n_steps_total=len(steps), n_steps_with_memory=len(memory_lens),
        n_steps_with_reflection=len(reflection_lens), n_steps_with_plan=len(plan_lens),
        mean_memory_length=mean_memory, memory_proportion=memory_prop, memory_variability=memory_var,
        mean_reflection_length=mean_reflection, reflection_proportion=reflection_prop,
        reflection_variability=reflection_var,
        mean_plan_length=mean_plan, plan_proportion=plan_prop, plan_variability=plan_var,
        extraction_warnings=warnings_list,
    )

# %% [markdown]
# ## 4. Export — produce BOTH predictor tables from identical extraction/aggregation code

# %%
DATA_DIR = Path("/kaggle/input/datasets/samuelstephen77/reasonstate-dataset/data")



  # adjust if Kaggle dataset is mounted elsewhere, e.g. /kaggle/input/...

PREDICTOR_FIELDS = [
    "mean_memory_length", "memory_proportion", "memory_variability",
    "mean_reflection_length", "reflection_proportion", "reflection_variability",
    "mean_plan_length", "plan_proportion", "plan_variability",
]

def load_labels():
    labels = {}
    for fname in ["alfworld_labels.json", "gaia_labels.json", "webshop_labels.json"]:
        with open(DATA_DIR / fname, "r", encoding="utf-8") as f:
            entries = json.load(f)
        for entry in entries:
            labels[entry["trajectory_id"]] = entry
    return labels


def build_table(trajectories, labels, tokenizer_fn):
    rows = []
    unmatched = []
    for traj in trajectories:
        preds = compute_predictors(traj, tokenizer_fn)
        label = labels.get(traj.trajectory_id)
        if label is None:
            unmatched.append(traj.trajectory_id)
            continue
        row = {
            "trajectory_id": traj.trajectory_id, "environment": traj.environment,
            "model": preds.model, "n_steps_total": preds.n_steps_total,
            "critical_failure_module": label.get("critical_failure_module"),
            "failure_type": (label["step_annotations"][0][label.get("critical_failure_module")].get("failure_type")
                             if label.get("step_annotations") else None),
            "critical_failure_step": label.get("critical_failure_step"),
        }
        for f_name in PREDICTOR_FIELDS:
            row[f_name] = getattr(preds, f_name)
        row["extraction_warnings"] = ";".join(preds.extraction_warnings)
        row["parse_errors"] = ";".join(traj.parse_errors)
        rows.append(row)
    return rows, unmatched


def write_csv(rows, path):
    fieldnames = (["trajectory_id", "environment", "model", "n_steps_total",
                    "critical_failure_module", "failure_type", "critical_failure_step"]
                  + PREDICTOR_FIELDS + ["extraction_warnings", "parse_errors"])
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# %%
trajectories = parse_all_trajectories(DATA_DIR)
labels = load_labels()

n_parse_errors = sum(1 for t in trajectories if t.parse_errors)
print(f"Parsed {len(trajectories)} trajectories. Parse errors: {n_parse_errors}")
assert len(trajectories) == 200, "Expected 200 trajectories per Blueprint v2.0 -- STOP if this fails."
assert n_parse_errors == 0, "Parse errors detected -- STOP. Document per Step 8B before proceeding."

# %%
rows_tiktoken, unmatched_tiktoken = build_table(trajectories, labels, count_tokens)
rows_diagnostic, unmatched_diagnostic = build_table(trajectories, labels, count_tokens_DIAGNOSTIC_ONLY)

print(f"Unmatched (tiktoken run): {len(unmatched_tiktoken)}")
print(f"Unmatched (diagnostic run): {len(unmatched_diagnostic)}")
assert len(unmatched_tiktoken) == 0 and len(unmatched_diagnostic) == 0, \
    "STOP -- label linkage failure. Do not proceed to Step 5."

write_csv(rows_tiktoken, "predictor_table_tiktoken.csv")
write_csv(rows_diagnostic, "predictor_table_diagnostic.csv")
print("Wrote predictor_table_tiktoken.csv and predictor_table_diagnostic.csv")

# %% [markdown]
# ## 5. Tokenizer comparison report
# Confirms the tokenizer swap changes scale (magnitude of length-based
# predictors), not extraction logic (which trajectories have which tags
# present, proportions, warnings). This is the evidence referenced in
# PIPELINE_V1.0_FROZEN.md justifying that sandbox-stage verification with
# the diagnostic tokenizer remains valid evidence about extraction
# correctness, independent of the final token-count scale.

# %%
import csv as _csv

def load_table(path):
    with open(path) as f:
        return {r["trajectory_id"]: r for r in _csv.DictReader(f)}

tt = load_table("predictor_table_tiktoken.csv")
dg = load_table("predictor_table_diagnostic.csv")

report_lines = []
report_lines.append("# Tokenizer Comparison Report\n")
report_lines.append(f"Trajectories compared: {len(tt)}\n")

# 1. Structural agreement: do both tokenizers agree on WHICH predictors are None
# (tag absence) vs present? This must be IDENTICAL regardless of tokenizer,
# since presence/absence comes from the parser, not the tokenizer.
structural_mismatches = []
for tid in tt:
    for field_name in PREDICTOR_FIELDS:
        tt_is_none = (tt[tid][field_name] == "")
        dg_is_none = (dg[tid][field_name] == "")
        if tt_is_none != dg_is_none:
            structural_mismatches.append((tid, field_name))

report_lines.append(f"\nStructural (presence/absence) mismatches: {len(structural_mismatches)}\n")
if structural_mismatches:
    report_lines.append("STOP -- this should be impossible if tokenizer only affects scale. "
                         "Investigate before proceeding.\n")
    for tid, fn in structural_mismatches[:20]:
        report_lines.append(f"  {tid}: {fn}\n")
else:
    report_lines.append("PASS: identical presence/absence pattern under both tokenizers, "
                         "confirming the tokenizer swap affects only magnitude, not extraction logic.\n")

# 2. Magnitude correlation for the length-based predictors
import numpy as np
length_fields = ["mean_memory_length", "mean_reflection_length", "mean_plan_length"]
report_lines.append("\n## Magnitude comparison (length predictors)\n")
for field_name in length_fields:
    pairs = [(float(tt[tid][field_name]), float(dg[tid][field_name]))
             for tid in tt if tt[tid][field_name] != "" and dg[tid][field_name] != ""]
    if len(pairs) >= 2:
        tt_vals, dg_vals = zip(*pairs)
        corr = np.corrcoef(tt_vals, dg_vals)[0, 1]
        ratio = np.mean(np.array(tt_vals) / np.array(dg_vals))
        report_lines.append(f"  {field_name}: n={len(pairs)}, Pearson r={corr:.4f}, "
                             f"mean(tiktoken/diagnostic ratio)={ratio:.3f}\n")

with open("tokenizer_comparison_report.md", "w") as f:
    f.writelines(report_lines)

print("".join(report_lines))

# %% [markdown]
# ## 6. Checksums (for the final Analysis Lock Certificate)

# %%
def sha256_of_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

for fname in ["predictor_table_tiktoken.csv", "predictor_table_diagnostic.csv",
              "tokenizer_comparison_report.md"]:
    print(f"{fname}: {sha256_of_file(fname)}")

print()
print("Python / package versions:")
import sys, scipy, skbio
print("Python:", sys.version)
print("tiktoken:", tiktoken.__version__ if hasattr(tiktoken, '__version__') else "version attr unavailable")
print("scipy:", scipy.__version__)
print("scikit-bio:", skbio.__version__)

# %% [markdown]
# ## STOP — Do not proceed to Step 5A in this notebook.
#
# Next actions (outside this notebook, per the frozen Stage 4 plan):
# 1. Review the structural-mismatch check above. If it did not PASS, stop
#    and investigate before anything else.
# 2. Complete Step 8C (predictor distributions, missingness, VIF, correlation
#    matrix) using predictor_table_tiktoken.csv.
# 3. Complete Step 8D (full reproducibility record) and Step 8E (readiness
#    checklist) using the version numbers and checksums printed above.
# 4. Produce the FINAL Analysis Lock Certificate.
# 5. Only then run Step 5A (the first PERMANOVA) -- in a separate,
#    clearly-labeled analysis notebook, not appended to this one.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 60.1 MB/s eta 0:00:00:00:01:01m
tiktoken o200k_base loaded OK. Vocab size: 200019
Parsed 200 trajectories. Parse errors: 0
Unmatched (tiktoken run): 0
Unmatched (diagnostic run): 0
Wrote predictor_table_tiktoken.csv and predictor_table_diagnostic.csv
# Tokenizer Comparison Report
Trajectories compared: 200

Structural (presence/absence) mismatches: 0
PASS: identical presence/absence pattern under both tokenizers, confirming the tokenizer swap affects only magnitude, not extraction logic.

## Magnitude comparison (length predictors)
  mean_memory_length: n=185, Pearson r=0.9972, mean(tiktoken/diagnostic ratio)=1.318
  mean_reflection_length: n=190, Pearson r=0.9641, mean(tiktoken/diagnostic ratio)=1.182
  mean_plan_length: n=192, Pearson r=0.9797, mean(tiktoken/diagno

In [2]:
import hashlib
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

files = [
    "paper13_pipeline_notebook.py",  # or ReasonState_pipeline.py if renamed per the codename convention
    "predictor_table_tiktoken.csv",
    "predictor_table_diagnostic.csv",
    "tokenizer_comparison_report.md",
]
for f in files:
    print(f"{f}: {sha256(f)}")

FileNotFoundError: [Errno 2] No such file or directory: 'paper13_pipeline_notebook.py'

In [4]:
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

files = [
    "/kaggle/working/.virtual_documents/__notebook_source__.ipynb",
    "predictor_table_tiktoken.csv",
    "predictor_table_diagnostic.csv",
    "tokenizer_comparison_report.md",
]

for f in files:
    try:
        print(f"{f}: {sha256(f)}")
    except FileNotFoundError:
        print(f"{f}: NOT FOUND")

/kaggle/working/.virtual_documents/__notebook_source__.ipynb: 8fc5ef96c2c626896c0a5e397e0317721148864654e534188566305ede932fe6
predictor_table_tiktoken.csv: 92eb46b762623e57af21d64941bd6514753aada886ad70e095801ebb1462f611
predictor_table_diagnostic.csv: 5788b852df67c65979c400351d9af95d155fc3e41269e070ffb715ad4dfb7f7d
tokenizer_comparison_report.md: a546b17b978020f201720a10dee4addb33fc35f9664e49771023ea2c8de7cf00


In [6]:
# %% [markdown]
# ## Step 5A — Confirmatory PERMANOVA (RQ1)
#
# Blueprint v2.0 Final, Analysis Lock Certificate signed. No further changes
# to predictors, outcomes, or statistical procedure permitted past this
# point except via formal governance reopening.
#
# Confirmatory predictor set (6): Memory Expression (mean length, proportion,
# variability) + Reflection Expression (mean length, proportion, variability).
# Outcome: critical_failure_module (6 levels, as released).
# Design: within-environment z-scoring, Euclidean distance, blocked
# permutation within environment, 9999 permutations, PERMDISP check,
# alpha=0.05.

# %%
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova, permdisp

from rse_statistics import blocked_permanova
 # validated module, see VALIDATION_LOG.md

PERMANOVA_SEED = 42  # recorded in the certificate addendum

df = pd.read_csv("predictor_table_tiktoken.csv")

CONFIRMATORY_PREDICTORS = [
    "mean_memory_length", "memory_proportion", "memory_variability",
    "mean_reflection_length", "reflection_proportion", "reflection_variability",
]

df_complete = df.dropna(subset=CONFIRMATORY_PREDICTORS).copy()
print(f"Complete cases for RQ1: {len(df_complete)} / {len(df)}")

modules = df_complete["critical_failure_module"].tolist()
envs = df_complete["environment"].values
X = df_complete[CONFIRMATORY_PREDICTORS].values.astype(float)

# %%
# Within-environment standardization (frozen Step 5A rule)
def within_env_zscore(X, envs):
    X_z = np.zeros_like(X, dtype=float)
    for env in np.unique(envs):
        mask = envs == env
        sub = X[mask]
        mean = sub.mean(axis=0)
        std = sub.std(axis=0)
        std[std == 0] = 1
        X_z[mask] = (sub - mean) / std
    return X_z

X_z = within_env_zscore(X, envs)

dist = squareform(pdist(X_z, metric="euclidean"))
ids = [f"t{i}" for i in range(len(df_complete))]
dm = DistanceMatrix(dist, ids=ids)

# %%
# Assumption check 1: PERMDISP (homogeneity of multivariate dispersion)
print("=== PERMDISP (dispersion homogeneity across module groups) ===")
permdisp_result = permdisp(dm, modules, permutations=9999)
print(permdisp_result)

# %%
# Assumption check 2: predictor redundancy (already audited in Step 8C;
# recorded here for completeness, not re-decided)
print("=== Predictor correlation matrix (see Step 6 threat log for VIF/Mantel discussion) ===")
print(df[CONFIRMATORY_PREDICTORS].corr().round(3))

# %%
# PRIMARY CONFIRMATORY TEST — RQ1
print("Running blocked PERMANOVA (9999 permutations)...")
observed_F, p_value, perm_distribution = blocked_permanova(
    dm, modules, envs, permutations=9999, seed=PERMANOVA_SEED
)

print()
print("=== RQ1 — PRIMARY CONFIRMATORY RESULT ===")
print(f"Observed pseudo-F: {observed_F:.4f}")
print(f"Blocked-permutation p-value: {p_value:.4f}")
print(f"Permutations: 9999, seed: {PERMANOVA_SEED}")
print(f"N (complete cases): {len(df_complete)}")

# %%
# Effect size: R^2 from the standard PERMANOVA decomposition
standard_result = permanova(dm, modules, permutations=0)
print("=== Effect size (R^2) ===")
print(standard_result)

# %% [markdown]
# ## Interpretation commitment (frozen, Step 5A)
#
# A significant PERMANOVA result establishes only the existence of an
# overall multivariate association between the Reasoning-State Expression
# profile and critical failure module. It does not identify which
# predictors or constructs drive that association — that is RQ2's role
# (Step 5B), not decided here.
#
# ## STOP
# Record this result before proceeding to RQ2/RQ3/Step 5D. Per Evidence
# Before Narrative: report whatever outcome occurred (O1/O2/O3/O4 per
# Step 7) without redesigning the analysis based on whether this result
# is the one you hoped for.

ModuleNotFoundError: No module named 'rse_statistics'

In [3]:
# %%
!pip install -q scikit-bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 31.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 46.7 MB/s eta 0:00:0000:0100:01


In [6]:
# %%
import numpy as np
from skbio.stats.distance import permanova

def blocked_permanova(dm, groups, blocks, permutations=9999, seed=42):
    """Blocked-permutation PERMANOVA. See VALIDATION_LOG.md for the
    validation suite this function passed before use in Step 5A."""
    rng = np.random.default_rng(seed)
    groups = np.array(groups)
    blocks = np.array(blocks)

    obs = permanova(dm, groups, permutations=0)
    observed_stat = obs["test statistic"]

    perm_stats = np.zeros(permutations)
    unique_blocks = np.unique(blocks)
    for p in range(permutations):
        permuted = groups.copy()
        for block in unique_blocks:
            mask = blocks == block
            idx = np.where(mask)[0]
            permuted[idx] = rng.permutation(groups[mask])
        res = permanova(dm, permuted, permutations=0)
        perm_stats[p] = res["test statistic"]

    p_value = (np.sum(perm_stats >= observed_stat) + 1) / (permutations + 1)
    return observed_stat, p_value, perm_stats

In [7]:
print("Cell executed successfully")
try:
    import skbio
    print("skbio imported OK, version:", skbio.__version__)
except ImportError as e:
    print("skbio import FAILED:", e)

try:
    print("blocked_permanova is defined:", blocked_permanova)
except NameError:
    print("blocked_permanova is NOT defined yet")

Cell executed successfully
skbio imported OK, version: 0.7.3
blocked_permanova is defined: <function blocked_permanova at 0x78d96475c9a0>


In [8]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova, permdisp

PERMANOVA_SEED = 42

df = pd.read_csv("predictor_table_tiktoken.csv")

CONFIRMATORY_PREDICTORS = [
    "mean_memory_length", "memory_proportion", "memory_variability",
    "mean_reflection_length", "reflection_proportion", "reflection_variability",
]

df_complete = df.dropna(subset=CONFIRMATORY_PREDICTORS).copy()
print(f"Complete cases for RQ1: {len(df_complete)} / {len(df)}")

modules = df_complete["critical_failure_module"].tolist()
envs = df_complete["environment"].values
X = df_complete[CONFIRMATORY_PREDICTORS].values.astype(float)

def within_env_zscore(X, envs):
    X_z = np.zeros_like(X, dtype=float)
    for env in np.unique(envs):
        mask = envs == env
        sub = X[mask]
        mean = sub.mean(axis=0)
        std = sub.std(axis=0)
        std[std == 0] = 1
        X_z[mask] = (sub - mean) / std
    return X_z

X_z = within_env_zscore(X, envs)

dist = squareform(pdist(X_z, metric="euclidean"))
ids = [f"t{i}" for i in range(len(df_complete))]
dm = DistanceMatrix(dist, ids=ids)

print("=== PERMDISP (dispersion homogeneity across module groups) ===")
permdisp_result = permdisp(dm, modules, permutations=9999)
print(permdisp_result)

print()
print("=== Predictor correlation matrix ===")
print(df[CONFIRMATORY_PREDICTORS].corr().round(3))

print()
print("Running blocked PERMANOVA (9999 permutations)...")
observed_F, p_value, perm_distribution = blocked_permanova(
    dm, modules, envs, permutations=9999, seed=PERMANOVA_SEED
)

print()
print("=== RQ1 — PRIMARY CONFIRMATORY RESULT ===")
print(f"Observed pseudo-F: {observed_F:.4f}")
print(f"Blocked-permutation p-value: {p_value:.4f}")
print(f"Permutations: 9999, seed: {PERMANOVA_SEED}")
print(f"N (complete cases): {len(df_complete)}")

print()
print("=== Effect size (R^2) ===")
standard_result = permanova(dm, modules, permutations=0)
print(standard_result)

Complete cases for RQ1: 182 / 200
=== PERMDISP (dispersion homogeneity across module groups) ===
method name               PERMDISP
test statistic name        F-value
sample size                    182
number of groups                 6
test statistic            2.047868
p-value                     0.0629
number of permutations        9999
Name: PERMDISP results, dtype: object

=== Predictor correlation matrix ===
                        mean_memory_length  memory_proportion  \
mean_memory_length                   1.000              0.774   
memory_proportion                    0.774              1.000   
memory_variability                   0.915              0.674   
mean_reflection_length               0.314             -0.225   
reflection_proportion               -0.593             -0.509   
reflection_variability              -0.208             -0.237   

                        memory_variability  mean_reflection_length  \
mean_memory_length                   0.915              

In [9]:
print(standard_result)
print()
print("Full result object attributes/keys available:")
print(standard_result.index.tolist() if hasattr(standard_result, 'index') else dir(standard_result))

method name               PERMANOVA
test statistic name        pseudo-F
sample size                     182
number of groups                  6
test statistic             3.207007
p-value                         NaN
number of permutations            0
Name: PERMANOVA results, dtype: object

Full result object attributes/keys available:
['method name', 'test statistic name', 'sample size', 'number of groups', 'test statistic', 'p-value', 'number of permutations']


In [10]:
import numpy as np

F = 3.207007
k = 6   # number of groups
N = 182 # sample size

df_between = k - 1
df_within = N - k

# F = [R^2/(k-1)] / [(1-R^2)/(N-k)]
# Solve for R^2:
R_squared = (F * df_between) / (F * df_between + df_within)

print(f"pseudo-F = {F}")
print(f"df_between = {df_between}, df_within = {df_within}")
print(f"R^2 = {R_squared:.4f}")

pseudo-F = 3.207007
df_between = 5, df_within = 176
R^2 = 0.0835


In [3]:
!pip install -q scikit-posthocs

In [5]:
CONFIRMATORY_PREDICTORS = [
    "mean_memory_length", "memory_proportion", "memory_variability",
    "mean_reflection_length", "reflection_proportion", "reflection_variability",
]

In [7]:
import pandas as pd
import numpy as np

CONFIRMATORY_PREDICTORS = [
    "mean_memory_length", "memory_proportion", "memory_variability",
    "mean_reflection_length", "reflection_proportion", "reflection_variability",
]

df = pd.read_csv("predictor_table_tiktoken.csv")

def within_env_zscore(X, envs):
    X_z = np.zeros_like(X, dtype=float)
    for env in np.unique(envs):
        mask = envs == env
        sub = X[mask]
        mean = sub.mean(axis=0)
        std = sub.std(axis=0)
        std[std == 0] = 1
        X_z[mask] = (sub - mean) / std
    return X_z

In [8]:
# %% [markdown]
# ## Step 5B — Exploratory Component Analysis (RQ2)
#
# RQ1 result is RECORDED, not reinterpreted here. This step decomposes
# which individual predictors associate with the outcome, using the same
# preprocessing as RQ1 (within-environment z-scoring). All 9 predictors
# (6 confirmatory + 3 Planning exploratory-only) are tested here, against
# BOTH outcomes per the frozen hierarchy: primary exploratory = module,
# secondary exploratory = failure_type. Two separate BH correction
# families, not pooled.

# %%
from scipy.stats import kruskal
import scikit_posthocs as sp  # for Dunn's test; install if unavailable
import numpy as np
import pandas as pd

ALL_PREDICTORS = CONFIRMATORY_PREDICTORS + [
    "mean_plan_length", "plan_proportion", "plan_variability",
]

df_5b = df.dropna(subset=ALL_PREDICTORS).copy()
print(f"Complete cases for RQ2: {len(df_5b)} / {len(df)}")

envs_5b = df_5b["environment"].values
X_5b = df_5b[ALL_PREDICTORS].values.astype(float)
X_5b_z = within_env_zscore(X_5b, envs_5b)

for i, col in enumerate(ALL_PREDICTORS):
    df_5b[col + "_z"] = X_5b_z[:, i]

# %%
def run_kruskal_family(df, predictors, outcome_col, label):
    results = []
    for pred in predictors:
        zcol = pred + "_z"
        groups = [df[df[outcome_col] == g][zcol].values for g in df[outcome_col].unique()]
        groups = [g for g in groups if len(g) > 0]
        stat, p = kruskal(*groups)
        results.append({"predictor": pred, "outcome": outcome_col, "H": stat, "p_raw": p})
    res_df = pd.DataFrame(results)
    # Benjamini-Hochberg correction within this family
    res_df = res_df.sort_values("p_raw").reset_index(drop=True)
    m = len(res_df)
    res_df["rank"] = np.arange(1, m + 1)
    res_df["p_bh"] = res_df["p_raw"] * m / res_df["rank"]
    res_df["p_bh"] = res_df["p_bh"].cummin()[::-1].cummin()[::-1]  # enforce monotonicity
    res_df["p_bh"] = res_df["p_bh"].clip(upper=1.0)
    res_df["significant_bh"] = res_df["p_bh"] < 0.05
    print(f"=== Family: {label} ===")
    print(res_df[["predictor", "H", "p_raw", "p_bh", "significant_bh"]].to_string(index=False))
    print()
    return res_df

print("=== RQ2 — Component-level Kruskal-Wallis tests ===\n")
family_module = run_kruskal_family(df_5b, ALL_PREDICTORS, "critical_failure_module", "Primary exploratory (module)")
family_failtype = run_kruskal_family(df_5b, ALL_PREDICTORS, "failure_type", "Secondary exploratory (failure_type)")

# %%
# Dunn's post-hoc for any predictor whose omnibus Kruskal-Wallis survived BH correction
print("=== Dunn's post-hoc (BH-adjusted) for BH-significant predictors only ===\n")
for _, row in family_module[family_module["significant_bh"]].iterrows():
    pred = row["predictor"]
    zcol = pred + "_z"
    print(f"--- {pred} (module) ---")
    dunn = sp.posthoc_dunn(df_5b, val_col=zcol, group_col="critical_failure_module", p_adjust="fdr_bh")
    print(dunn.round(4))
    print()

# %% [markdown]
# ## STOP — record RQ2 results before proceeding to RQ3 (Step 5C)

Complete cases for RQ2: 182 / 200
=== RQ2 — Component-level Kruskal-Wallis tests ===

=== Family: Primary exploratory (module) ===
             predictor         H    p_raw    p_bh  significant_bh
      mean_plan_length 20.424857 0.001040 0.00557            True
mean_reflection_length 20.011010 0.001244 0.00557            True
    mean_memory_length 19.080643 0.001857 0.00557            True
    memory_variability 16.652623 0.005208 0.00557            True
      plan_variability 14.409183 0.013209 0.00557            True
 reflection_proportion 12.624972 0.027158 0.00557            True
     memory_proportion 11.914193 0.035983 0.00557            True
reflection_variability 10.043084 0.074024 0.00557            True
       plan_proportion  7.590617 0.180288 0.00557            True

=== Family: Secondary exploratory (failure_type) ===
             predictor         H    p_raw     p_bh  significant_bh
      mean_plan_length 53.927673 0.000010 0.000058            True
mean_reflection_lengt

In [10]:
!pip install -q scikit-bio
from skbio.stats.distance import DistanceMatrix, permanova, permdisp
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform

CONFIRMATORY_PREDICTORS = [
    "mean_memory_length", "memory_proportion", "memory_variability",
    "mean_reflection_length", "reflection_proportion", "reflection_variability",
]

df = pd.read_csv("predictor_table_tiktoken.csv")

def within_env_zscore(X, envs):
    X_z = np.zeros_like(X, dtype=float)
    for env in np.unique(envs):
        mask = envs == env
        sub = X[mask]
        mean = sub.mean(axis=0)
        std = sub.std(axis=0)
        std[std == 0] = 1
        X_z[mask] = (sub - mean) / std
    return X_z

df_complete = df.dropna(subset=CONFIRMATORY_PREDICTORS).copy()
modules = df_complete["critical_failure_module"].tolist()
envs = df_complete["environment"].values
X = df_complete[CONFIRMATORY_PREDICTORS].values.astype(float)
X_z = within_env_zscore(X, envs)
dist = squareform(pdist(X_z, metric="euclidean"))
ids = [f"t{i}" for i in range(len(df_complete))]
dm = DistanceMatrix(dist, ids=ids)
print("Setup complete. N =", len(df_complete))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.2 MB/s eta 0:00:00:00:010:01
Setup complete. N = 182


In [11]:
# Step 5C — Environment Moderation (RQ3)
# Tests whether the Module x Environment interaction is significant.
# Uses the SAME distance matrix as RQ1 (dm), same 6-predictor profile.
# No new predictors introduced.

from skbio.stats.distance import permanova

print("Running factorial PERMANOVA: Module + Environment + Module x Environment...")
print("(This uses sequential permutation testing)")
print()

# Step 1: Module effect (already known from RQ1, reproduced here for the partition)
res_module = permanova(dm, modules, permutations=9999)
print("Module only:")
print(res_module)
print()

# Step 2: Environment effect
envs_list = df_complete["environment"].tolist()
res_env = permanova(dm, envs_list, permutations=9999)
print("Environment only:")
print(res_env)
print()

# Step 3: Combined grouping (Module x Environment interaction proxy)
# Create a combined group label: environment_module
combined = [f"{e}_{m}" for e, m in zip(envs_list, modules)]
res_combined = permanova(dm, combined, permutations=9999)
print("Module x Environment (combined):")
print(res_combined)
print()

# Interaction R^2 estimate:
# R^2_interaction = R^2_combined - R^2_module - R^2_environment
def r2_from_F(F, k, N):
    df_b = k - 1
    df_w = N - k
    return (F * df_b) / (F * df_b + df_w)

N = len(df_complete)
k_module = len(set(modules))
k_env = len(set(envs_list))
k_combined = len(set(combined))

r2_module = r2_from_F(res_module["test statistic"], k_module, N)
r2_env = r2_from_F(res_env["test statistic"], k_env, N)
r2_combined = r2_from_F(res_combined["test statistic"], k_combined, N)
r2_interaction = r2_combined - r2_module - r2_env

print(f"R^2 module:      {r2_module:.4f}")
print(f"R^2 environment: {r2_env:.4f}")
print(f"R^2 combined:    {r2_combined:.4f}")
print(f"R^2 interaction (estimate): {r2_interaction:.4f}")
print()
print(f"RQ3 interaction p-value comes from the combined model significance.")
print(f"Per frozen Step 5C: if interaction term p < 0.05, outcome = O2; else O1.")

Running factorial PERMANOVA: Module + Environment + Module x Environment...
(This uses sequential permutation testing)

Module only:
method name               PERMANOVA
test statistic name        pseudo-F
sample size                     182
number of groups                  6
test statistic             3.207007
p-value                      0.0001
number of permutations         9999
Name: PERMANOVA results, dtype: object

Environment only:
method name               PERMANOVA
test statistic name        pseudo-F
sample size                     182
number of groups                  3
test statistic                  0.0
p-value                         1.0
number of permutations         9999
Name: PERMANOVA results, dtype: object

Module x Environment (combined):
method name               PERMANOVA
test statistic name        pseudo-F
sample size                     182
number of groups                 16
test statistic             2.069026
p-value                      0.0002
number of permut

In [12]:
# Proper interaction test: does combined significantly improve on module alone?
# Compare R^2_combined vs R^2_module using a permutation F-test

r2_module = 0.0835
r2_combined = 0.1575
N = 182
k_combined = 16
k_module = 6

# Additional F from the interaction terms
df_interaction = k_combined - k_module  # 10 additional df
df_residual = N - k_combined  # 166

F_interaction = ((r2_combined - r2_module) / df_interaction) / ((1 - r2_combined) / df_residual)
print(f"F for interaction (given module): {F_interaction:.4f}")
print(f"df_interaction={df_interaction}, df_residual={df_residual}")

from scipy.stats import f as f_dist
p_parametric = 1 - f_dist.cdf(F_interaction, df_interaction, df_residual)
print(f"Parametric p (approximate): {p_parametric:.4f}")

F for interaction (given module): 1.4580
df_interaction=10, df_residual=166
Parametric p (approximate): 0.1594


In [13]:
# Step 5D — Sensitivity Analyses
# Four pre-specified checks. Per frozen rules: "Robust" if all four
# preserve the qualitative conclusion (association present/absent,
# direction unchanged). "Qualified robustness" if any one fails.

import numpy as np
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova

def r2_from_F(F, k, N):
    return (F * (k-1)) / (F * (k-1) + (N-k))

# --- Sensitivity 1: Unrestricted (unblocked) permutation ---
res_unblocked = permanova(dm, modules, permutations=9999)
print("S1 — Unrestricted permutation:")
print(f"  pseudo-F={res_unblocked['test statistic']:.4f}, p={res_unblocked['p-value']:.4f}")
print()

# --- Sensitivity 2: Gower distance instead of Euclidean ---
from sklearn.metrics import pairwise_distances
X_gower = X_z  # already standardized; Gower on standardized continuous = Manhattan/n
dist_gower = squareform(pairwise_distances(X_z, metric="l1")) / X_z.shape[1]
dm_gower = DistanceMatrix(dist_gower, ids=[f"t{i}" for i in range(len(df_complete))])
res_gower = permanova(dm_gower, modules, permutations=9999)
print("S2 — Gower (L1/n) distance:")
print(f"  pseudo-F={res_gower['test statistic']:.4f}, p={res_gower['p-value']:.4f}")
print()

# --- Sensitivity 3: plan/planning merged ---
df_merged = df_complete.copy()
df_merged["critical_failure_module"] = df_merged["critical_failure_module"].replace("plan", "planning")
modules_merged = df_merged["critical_failure_module"].tolist()
res_merged = permanova(dm, modules_merged, permutations=9999)
print("S3 — plan/planning merged:")
print(f"  pseudo-F={res_merged['test statistic']:.4f}, p={res_merged['p-value']:.4f}")
print()

# --- Sensitivity 4: Per-environment (no pooling) ---
print("S4 — Per-environment analyses:")
for env in ["ALFWorld", "GAIA", "WebShop"]:
    mask = df_complete["environment"] == env
    idx = np.where(mask)[0]
    if len(idx) < 10:
        print(f"  {env}: too few cases, skipped")
        continue
    sub_dist = squareform(pdist(X_z[idx], metric="euclidean"))
    sub_ids = [f"t{i}" for i in range(len(idx))]
    sub_dm = DistanceMatrix(sub_dist, sub_ids)
    sub_modules = [modules[i] for i in idx]
    if len(set(sub_modules)) < 2:
        print(f"  {env}: fewer than 2 module groups, skipped")
        continue
    res_env = permanova(sub_dm, sub_modules, permutations=9999)
    print(f"  {env}: pseudo-F={res_env['test statistic']:.4f}, p={res_env['p-value']:.4f}")

S1 — Unrestricted permutation:
  pseudo-F=3.2070, p=0.0001

S2 — Gower (L1/n) distance:
  pseudo-F=3.1701, p=0.0004

S3 — plan/planning merged:
  pseudo-F=3.6982, p=0.0001

S4 — Per-environment analyses:
  ALFWorld: pseudo-F=4.3735, p=0.0001
  GAIA: pseudo-F=0.4264, p=0.9419
  WebShop: pseudo-F=2.1088, p=0.0214
